# RoomBeacon — Bronze to Canonical Silver

This notebook performs deterministic cleaning, validation, and quality classification only.

**Canonical Silver Dataset:** `silver.rental_listings`  
**Deprecated / TEMPORARY COMPATIBILITY MIRROR:** `rental_latest.parquet`

No market analysis, recommendation, feature engineering, or model fitting belongs here.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'crawler').is_dir() and (p / 'analytics').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env', override=False)

import pandas as pd
from IPython.display import display, Markdown
from analytics.duckdb.connection import create_analytics_connection
from analytics.silver.materializer import SilverMaterializer
from notebooks.utils.notebook_audit import load_snapshot
from notebooks.utils.silver_processing import build_silver_dataset, evaluate_pre_silver_quality_gate

pd.set_option('display.max_columns', 20)


## 01 Load Bronze Snapshot

Read one runtime snapshot from the canonical Bronze-backed analytical view. The snapshot remains immutable.


In [2]:
conn = create_analytics_connection()
bronze_df, raw_evidence, RUN_CONTEXT = load_snapshot(conn, PROJECT_ROOT)
BRONZE_COLUMNS = bronze_df.columns.tolist()
display(pd.Series(RUN_CONTEXT, name='runtime_value').to_frame())


,runtime_value
snapshot_started_utc,2026-09-30T05:14:06.028701+00:00
snapshot_completed_utc,2026-09-30T05:14:32.484545+00:00
row_count,132378
field_count,20
source_count,9
view_name,v_latest_posts
grain,one row per rental_post_id
raw_grain,latest child ID of contributing version; post ...
max_latest_observed_at,2026-09-30 05:00:04


## 02 Processing Contract

- Grain: one row per `rental_post_id`.
- Preserve every Bronze row and raw/source field.
- Add clean values and explicit quality statuses.
- Never delete outliers or duplicate candidates and never guess missing values.
- Materialize DuckDB only after the quality gate passes.


In [3]:
assert len(bronze_df) > 0
assert bronze_df.rental_post_id.notna().all()
assert bronze_df.rental_post_id.is_unique
RAW_FINGERPRINT = bronze_df.copy(deep=True)
display(Markdown(f'**Bronze contract accepted:** {len(bronze_df):,} rows at one row per rental_post_id.'))


**Bronze contract accepted:** 132,378 rows at one row per rental_post_id.

## 03 General Text Standardization

Canonical text utilities normalize Unicode, invisible characters, whitespace, safe punctuation, and decorative icons while preserving raw columns.


In [4]:
silver_df = build_silver_dataset(bronze_df, raw_evidence)
text_clean_columns = ['title_clean', 'full_address_text_clean', 'location_raw_clean', 'best_address_text_clean']
display(silver_df[text_clean_columns].notna().sum().rename('non_null_count').to_frame())


,non_null_count
title_clean,131925
full_address_text_clean,114673
location_raw_clean,119032
best_address_text_clean,119202


## 04 Title Quality Processing

Keep `title_raw`; expose `title_clean` and a documented usability status.


In [5]:
display(silver_df.title_quality_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
USABLE,131277
TOO_SHORT,648
MISSING,453


## 05 Address Standardization & Parsing

Parse the best available cleaned address into traceable components without inventing an address.


In [6]:
display(silver_df.parse_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
PARTIALLY_PARSED,72582
PARSED,45132
NO_ADDRESS_EVIDENCE,13176
UNRECOGNIZED_FORMAT,1488


## 06 Administrative Unit Mapping

Apply the internal RoomBeacon ward reference. Ambiguous candidates remain unresolved and never become current ward truth.


In [7]:
display(silver_df.ward_mapping_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
UNCHANGED,99148
MISSING,19572
MAPPED,8792
UNMAPPED,4460
AMBIGUOUS,406


## 07 Price Validation

Preserve raw/current price evidence and add only validated existing values or deterministic, lineage-aligned recovery.


In [8]:
display(silver_df.price_quality_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
VALIDATED_EXISTING,128780
MISSING_OR_REVIEW,3595
REPARSE_ACCEPTED_CLEAN,3


## 08 Area Validation

Preserve raw/current area evidence and add only validated existing values or deterministic, lineage-aligned recovery.


In [9]:
display(silver_df.area_quality_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
VALIDATED_EXISTING,130266
MISSING_OR_REVIEW,2099
REPARSE_ACCEPTED_CLEAN,13


## 09 Numeric Outlier Flags

Statistical outliers remain in the dataset and receive explicit flags.


In [10]:
display(silver_df.numeric_outlier_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
NOT_OUTLIER,120170
PRICE_OUTLIER,6903
AREA_OUTLIER,4728
PRICE_AND_AREA_OUTLIER,577


## 10 Coordinate Trust Classification

Coordinates are usable only when the pair, bounds, provider, and repeated-point evidence pass the canonical audit.


In [11]:
display(silver_df.coordinate_quality_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
INVALID,119529
UNTRUSTED,10572
USABLE,2277


## 11 Cross-field Consistency

Classify price/area combinations without filtering rows or interpreting them as market truth.


In [12]:
display(silver_df.cross_field_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
CONSISTENT,126704
INSUFFICIENT_DATA,5674


## 12 Duplicate / Repost Candidate Classification

Fingerprint matches are evidence for review, not deletion or deduplication.


In [13]:
display(silver_df.duplicate_candidate_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
UNIQUE_FINGERPRINT,114779
INSUFFICIENT_DATA,13505
POSSIBLE_DUPLICATE,4094


## 13 Temporal Semantics Validation

Invalid or out-of-order observation timestamps are marked `REQUIRES_REVIEW`; no unverified backfill is performed.


In [14]:
display(silver_df.temporal_quality_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
VALID,131996
REQUIRES_REVIEW,382


## 14 Final Row Quality Status

Roll up critical review conditions and non-destructive flags into a row-level readiness status.


In [15]:
display(silver_df.row_quality_status.value_counts(dropna=False).rename_axis('status').to_frame('rows'))


,rows
status,
READY_WITH_FLAGS,128457
READY,2040
REQUIRES_REVIEW,1881


## 15 Silver Dataset Contract

`silver.rental_listings` preserves Bronze grain and raw fields and appends deterministic clean values, lineage, and quality statuses.


In [16]:
silver_contract = pd.Series({
    'rows': len(silver_df),
    'distinct_rental_post_id': silver_df.rental_post_id.nunique(),
    'columns': len(silver_df.columns),
    'raw_columns_preserved': set(BRONZE_COLUMNS).issubset(silver_df.columns),
}, name='value')
display(silver_contract.to_frame())


,value
rows,132378
distinct_rental_post_id,132378
columns,56
raw_columns_preserved,True


## 16 Pre-Silver Quality Gate

Every critical preservation and status invariant must pass. Failure stops execution before any canonical write.


In [17]:
quality_gate = evaluate_pre_silver_quality_gate(RAW_FINGERPRINT, silver_df)
quality_gate_frame = quality_gate.to_frame()
assert quality_gate.passed
display(quality_gate_frame)


,invariant,passed
0,row_count_preserved,True
1,rental_post_id_non_null,True
2,rental_post_id_unique,True
3,source_identity_preserved,True
4,raw_source_fields_unchanged,True
5,no_row_multiplication_or_deletion,True
6,price_clean_contract_valid,True
7,area_clean_contract_valid,True
8,ambiguous_ward_not_truth,True
9,invalid_coordinates_not_usable,True


## 17 Materialize silver.rental_listings

Publish order is fixed: quality gate → canonical DuckDB table → canonical validation → TEMPORARY COMPATIBILITY MIRROR.


In [18]:
materializer = SilverMaterializer()
materialization = materializer.materialize(silver_df, conn=conn)
display(pd.Series(materialization.__dict__, name='value').to_frame())


,value
generated_at,2026-09-30T12:15:15.328167+07:00
row_count,132378
unique_listing_count,132378
canonical_table,silver.rental_listings
compatibility_mirror,rental_latest.parquet
mirror_status,DEPRECATED_TEMPORARY_COMPATIBILITY_MIRROR
min_observed_at,2026-09-20 12:48:44
max_observed_at,2026-09-30 05:00:04
schema_version,2.0.0
materializer_version,2.0.0


## 18 Silver Validation Summary

Validate the persisted canonical table and confirm identical grain and columns in the compatibility mirror.


In [19]:
canonical_summary = conn.execute("""
    SELECT count(*) AS row_count,
           count(DISTINCT rental_post_id) AS distinct_rental_post_id
    FROM silver.rental_listings
""").df()
canonical_columns = [row[0] for row in conn.execute('DESCRIBE silver.rental_listings').fetchall()]
mirror_path = str(materializer.output_file).replace("'", "''")
mirror_summary = conn.execute(f"""
    SELECT count(*) AS row_count,
           count(DISTINCT rental_post_id) AS distinct_rental_post_id
    FROM read_parquet('{mirror_path}')
""").df()
mirror_columns = [row[0] for row in conn.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{mirror_path}')"
).fetchall()]
assert canonical_summary.equals(mirror_summary)
assert canonical_columns == mirror_columns
assert canonical_summary.loc[0, 'row_count'] == canonical_summary.loc[0, 'distinct_rental_post_id']
display(canonical_summary.assign(column_count=len(canonical_columns), materialization_status='PASS'))
display(Markdown('**Canonical Silver and compatibility mirror have identical grain, row count, and columns.**'))
conn.close()


,row_count,distinct_rental_post_id,column_count,materialization_status
0,132378,132378,56,PASS


**Canonical Silver and compatibility mirror have identical grain, row count, and columns.**